In [1]:
import pandas as pd
import numpy as np

# Load the avg_by_model_and_classifier.csv file
df = pd.read_csv('avg_by_model_and_classifier.csv')

def process_split(df, mask_frac_value, output_file):
    df_filtered = df[df['mask_frac'] == mask_frac_value]
    
    def compute_avg_pm(series):
        means = []
        stds = []
        for item in series:
            mean_str, std_str = item.split('±')
            means.append(float(mean_str.strip()))
            stds.append(float(std_str.strip()))
        avg_mean = sum(means) / len(means)
        avg_std = sum(stds) / len(stds)
        return f"{avg_mean:.4f} ± {avg_std:.4f}"
    
    # Group by classifier and embedding, compute aggregated averages
    df_grouped = df_filtered.groupby(['classifier', 'embedding']).agg({
        'accuracy_pm': compute_avg_pm,
        'f1_pm': compute_avg_pm,
    }).reset_index()
    
    # Rename columns as required
    df_grouped.rename(columns={
        'classifier': 'Classifier',
        'embedding': 'Embedding',
        'accuracy_pm': 'Accuracy',
        'f1_pm': 'F1'
    }, inplace=True)
    
    # Save the result
    df_grouped.to_csv(output_file, index=False)

# Process for 30-70 split (mask_frac = 0.3)
process_split(df, 0.3, 'analysis_results_30_70.csv')

# Process for 70-30 split (mask_frac = 0.7)
process_split(df, 0.7, 'analysis_results_70_30.csv')

In [2]:
import pandas as pd

# Load the file
df = pd.read_csv('avg_by_model_and_classifier.csv')

def process_split_datasetwise(df, mask_frac_value, output_file):
    # Filter for the chosen mask fraction
    df_filtered = df[df['mask_frac'] == mask_frac_value]

    def compute_avg_pm(series):
        means = []
        stds = []
        for item in series:
            mean_str, std_str = item.split('±')
            means.append(float(mean_str.strip()))
            stds.append(float(std_str.strip()))
        avg_mean = sum(means) / len(means)
        avg_std = sum(stds) / len(stds)
        return f"{avg_mean:.4f} ± {avg_std:.4f}"

    # Group by classifier, embedding, dataset
    df_grouped = df_filtered.groupby(['classifier', 'embedding', 'dataset']).agg({
        'accuracy_pm': compute_avg_pm
    }).reset_index()

    # Pivot so datasets become columns
    pivot_acc = df_grouped.pivot_table(
        index=['classifier', 'embedding'],
        columns='dataset',
        values='accuracy_pm',
        aggfunc='first'
    )

    # Flatten multiindex column names
    pivot_acc.columns = [f"Accuracy_{col}" for col in pivot_acc.columns]

    # Add "Average across datasets" column
    def avg_pm_row(row):
        means, stds = [], []
        for val in row.dropna():
            mean_str, std_str = val.split('±')
            means.append(float(mean_str.strip()))
            stds.append(float(std_str.strip()))
        return f"{sum(means)/len(means):.4f} ± {sum(stds)/len(stds):.4f}" if means else ""

    pivot_acc["Accuracy_Avg"] = pivot_acc.apply(avg_pm_row, axis=1)

    # Add "Average across models" row
    avg_row = {}
    for col in pivot_acc.columns:
        if col in ["classifier", "embedding"]:
            continue
        means, stds = [], []
        for val in pivot_acc[col].dropna():
            mean_str, std_str = val.split('±')
            means.append(float(mean_str.strip()))
            stds.append(float(std_str.strip()))
        avg_row[col] = f"{sum(means)/len(means):.4f} ± {sum(stds)/len(stds):.4f}" if means else ""

    avg_df = pd.DataFrame([avg_row], index=[("Average", "Average")])

    # Concatenate
    final_df = pd.concat([pivot_acc, avg_df])
    final_df = final_df.reset_index()

    # Save
    final_df.to_csv(output_file, index=False)

    print(f"Saved {output_file}")
    return final_df

# Example usage
results_30_70 = process_split_datasetwise(df, 0.3, 'datasetwise_results_30_70.csv')
results_70_30 = process_split_datasetwise(df, 0.7, 'datasetwise_results_70_30.csv')


Saved datasetwise_results_30_70.csv
Saved datasetwise_results_70_30.csv


In [3]:
import pandas as pd

# Load the CSV data
df = pd.read_csv('avg_embedding_times.csv')

# Filter out 'random' and 'given' embeddings
df = df[~df['embedding'].isin(['random', 'given'])]

# Create a mapping for better display names
embedding_name_map = {
    'deepwalk': 'DeepWalk',
    'node2vec': 'Node2Vec',
    'vgae': 'VGAE',
    'dgi': 'DGI',
    'fuse': 'FUSE'
}
df['embedding'] = df['embedding'].map(embedding_name_map)

# Create a new column to represent the split type
df['split'] = df['mask_frac'].apply(lambda x: '70-30' if x == 0.7 else '30-70')

# Pivot the data to get dataset × split rows and embeddings as columns
pivot_df = df.pivot_table(
    index=['dataset', 'split'],
    columns='embedding',
    values='avg_embedding_time',
    aggfunc='mean'
).reset_index()

# Rename datasets
dataset_name_map = {
    'cora': 'Cora',
    'citeseer': 'CiteSeer',
    'photo': 'Amazon-Photo',
    'wikics': 'WikiCS',
    'pubmed': 'PubMed'
}
pivot_df['dataset'] = pivot_df['dataset'].map(dataset_name_map)

# Ensure embedding columns exist (even if some are missing) and define order
embed_cols = ['DeepWalk', 'Node2Vec', 'VGAE', 'DGI', 'FUSE']
for col in embed_cols:
    if col not in pivot_df.columns:
        pivot_df[col] = pd.NA

# Reorder columns
column_order = ['dataset', 'split'] + embed_cols
pivot_df = pivot_df[column_order]

# Convert embedding columns to numeric (coerce missing strings -> NaN)
pivot_df[embed_cols] = pivot_df[embed_cols].apply(pd.to_numeric, errors='coerce')

# --- ROW-WISE: add column that averages across embedding columns for each (dataset, split) ---
pivot_df['Avg_all_embeddings'] = pivot_df[embed_cols].mean(axis=1)

# --- COLUMN-WISE: add average row per split (average across datasets for each split) ---
overall_avg_by_split = (
    pivot_df.groupby('split')[embed_cols + ['Avg_all_embeddings']]
    .mean()
    .reset_index()
)
overall_avg_by_split['dataset'] = 'Average'
# place columns in same order
overall_avg_by_split = overall_avg_by_split[['dataset', 'split'] + embed_cols + ['Avg_all_embeddings']]

# --- GLOBAL AVERAGE: average across both splits and all datasets ---
global_avg = pivot_df[embed_cols + ['Avg_all_embeddings']].mean().to_frame().T
global_avg['split'] = 'All'
global_avg['dataset'] = 'Average'
global_avg = global_avg[['dataset', 'split'] + embed_cols + ['Avg_all_embeddings']]

# --- Combine original rows + per-split average rows + global average row ---
final_df = pd.concat([pivot_df, overall_avg_by_split, global_avg], ignore_index=True)

# Optional: sort so real datasets come first, and average rows at bottom
# Keep the 'Average' dataset rows at the end
final_df['__is_avg'] = final_df['dataset'].eq('Average')
final_df = final_df.sort_values(['__is_avg', 'dataset', 'split']).drop(columns='__is_avg').reset_index(drop=True)

# Round numeric columns for readability
for col in embed_cols + ['Avg_all_embeddings']:
    final_df[col] = final_df[col].round(2)

# Save
final_df.to_csv('formatted_runtime_table_with_average.csv', index=False)

print("Saved 'formatted_runtime_table_with_average.csv'")
print(final_df.tail(8))


Saved 'formatted_runtime_table_with_average.csv'
embedding  dataset  split  DeepWalk  Node2Vec    VGAE     DGI    FUSE  \
5             Cora  70-30     50.48     47.26   12.95    6.78   12.52   
6           PubMed  30-70    477.77    448.58  226.29   36.05  109.15   
7           PubMed  70-30    490.72    453.74  235.24   39.43   95.79   
8           WikiCS  30-70    792.11    785.70  338.10  126.80  128.92   
9           WikiCS  70-30    747.20    745.33  329.46  134.58   86.45   
10         Average  30-70    333.14    324.68  145.58   46.08   66.27   
11         Average  70-30    326.42    316.99  145.85   48.43   51.52   
12         Average    All    329.78    320.83  145.71   47.26   58.89   

embedding  Avg_all_embeddings  
5                       26.00  
6                      259.57  
7                      262.98  
8                      434.32  
9                      408.60  
10                     183.15  
11                     177.84  
12                     180.50  


In [4]:
import pandas as pd

# Load your file
df = pd.read_csv("per_run_results_all.csv")

# Filter only classifiers of interest
df_filtered = df[df["classifier"].isin(["gat", "gcn", "graphsage"])]

# Add new column for total time
df_filtered["total_time_seconds"] = (
    df_filtered["embedding_time_seconds"] + df_filtered["train_time_seconds"]
)

def make_split_table(df, mask_frac_value, output_file):
    # Filter for a given split (mask_frac)
    df_split = df[df["mask_frac"] == mask_frac_value]

    # Group by embedding, classifier, dataset
    dataset_wise = (
        df_split.groupby(["embedding", "classifier", "dataset"])["total_time_seconds"]
        .mean()
        .reset_index()
    )

    # Pivot so datasets become columns
    pivot_table = dataset_wise.pivot_table(
        index=["embedding", "classifier"],
        columns="dataset",
        values="total_time_seconds"
    ).reset_index()

    # Add average across datasets (row-wise)
    pivot_table["Avg_all_datasets"] = pivot_table.drop(columns=["embedding", "classifier"]).mean(axis=1)

    # Add average row (column-wise)
    avg_row = pivot_table.drop(columns=["embedding", "classifier"]).mean()
    avg_row["embedding"] = "Average"
    avg_row["classifier"] = "Average"

    # Append average row
    pivot_table = pd.concat([pivot_table, avg_row.to_frame().T], ignore_index=True)

    # Save
    pivot_table.to_csv(output_file, index=False)
    print(f"Saved {output_file}")
    return pivot_table

# Generate tables for both splits
table_30_70 = make_split_table(df_filtered, 0.3, "dataset_wise_total_times_30_70.csv")
table_70_30 = make_split_table(df_filtered, 0.7, "dataset_wise_total_times_70_30.csv")


Saved dataset_wise_total_times_30_70.csv
Saved dataset_wise_total_times_70_30.csv
